**USD-collateralized JPY curve (CSA-based)**

In [1]:
# Sofr data
sfSwRT= [('depo', '1d', 3.60),
      ('swap','1w', 3.66154),('swap','2w', 3.68178),('swap','3w', 3.67395),
      ('swap','1m', 3.6731), ('swap','2m', 3.6794), ('swap','3m', 3.67135),
      ('swap','4m', 3.65985),('swap','5m', 3.65005),('swap','6m', 3.6309),
      ('swap','7m', 3.6101), ('swap','8m', 3.59355),('swap','9m', 3.5725),
      ('swap','10m',3.55115),('swap','11m',3.5339), ('swap','12m',3.5188),
      ('swap','18m',3.43495),('swap','2y', 3.4272), ('swap','3y', 3.4609),
      ('swap','4y', 3.5215), ('swap','5y', 3.58825)]
# Tona data
tnSwRT = [('depo','1d', 0.728),
      ('swap','1w', 0.7277), ('swap','2w', 0.72875),('swap','3w', 0.72875),
      ('swap','1m', 0.72875),('swap','2m', 0.72956),('swap','3m', 0.7414),
      ('swap','4m', 0.76665),('swap','5m', 0.79104),('swap','6m', 0.81747),
      ('swap','7m', 0.84223),('swap','8m', 0.86397),('swap','9m', 0.8872),
      ('swap','10m',0.91146),('swap','11m',0.9338), ('swap','12m',0.9575),
      ('swap','15m',1.02),   ('swap','18m',1.0825), ('swap','2y', 1.2025),
      ('swap','3y', 1.3775), ('swap','4y', 1.51375),('swap','5y', 1.6275)]
# xccy Basis
xcyBSS = [('fxsw','1W',-9.43),('fxsw','1M', -44.85),('fxsw','2M',  -81.51),
        ('fxsw','3M',-121.85),('fxsw','4M',-160.19),('fxsw','5M', -198.75),
        ('fxsw','6M',-234.62),('fxsw','9M',-337.59),('fxsw','12M',-433.70),

        ('bss','18m',-25.625),('bss','2y',-28.125), ('bss','3y',-31.375),
        ('bss','4y', -34.25), ('bss','5y',-36.5)]

In [2]:
from myABBR import * ; import myUTIL as mu

trdDT = jDT(2026,1,20);       calUJ=ql.JointCalendar(calUSf, calJP)
stlDT = adD(calUJ,trdDT,Tp2); setEvDT(trdDT)
# single crv
sfIX, sfCvOBJ, sfCvHDL, sfParRT = mu.makeSofrCurve(sfSwRT)
tnIX, tnCvOBJ, tnCvHDL, tnParRT = mu.makeTonaCurve(tnSwRT)
mu.showCvDT(sfCvOBJ,sfIX); dfDSP(mu.dfNodes(sfCvOBJ,sfParRT,cmpd=CMP),6)
mu.showCvDT(sfCvOBJ,tnIX); dfDSP(mu.dfNodes(tnCvOBJ,tnParRT,cmpd=CNT),3)

tradeDT:2026-01-20, settleDT:2026-01-20 (index:SOFRON Actual/360 )


,date,matYR,parRT,zeroRT,DF
0,2026-01-20,0.0000,nan%,3.665398%,1.00000000
1,2026-01-21,0.0028,3.600000%,3.665398%,0.99990001
2,2026-01-29,0.0250,3.661540%,3.721086%,0.99908704
3,2026-02-05,0.0444,3.681780%,3.741295%,0.99836888
4,2026-02-12,0.0639,3.673950%,3.734588%,0.99766023
5,2026-02-23,0.0944,3.673100%,3.732922%,0.99654466
16,2027-01-22,1.0194,3.518800%,3.518924%,0.96535756
17,2027-07-22,1.5222,3.434950%,3.443916%,0.94976419
18,2028-01-24,2.0389,3.427200%,3.425388%,0.93363441
19,2029-01-22,3.0500,3.460900%,3.460585%,0.90143926


tradeDT:2026-01-20, settleDT:2026-01-20 (index:TonarON Actual/365 (Fixed) )


,date,matYR,parRT,zeroRT,DF
0,2026-01-20,0.0000,nan%,0.727993%,1.00000000
1,2026-01-21,0.0027,0.728000%,0.727993%,0.99998006
2,2026-01-29,0.0247,0.727700%,0.727687%,0.99982059
20,2029-01-22,3.0082,1.377500%,1.370440%,0.95961239
21,2030-01-22,4.0082,1.513750%,1.507337%,0.94137154
22,2031-01-22,5.0082,1.627500%,1.622316%,0.92196397


In [3]:
# xcBasis crv  ( base: sofr )
jpySP,  baseIX, othrIX, clltCv, baseCLLT, baseBSS, baseRST, cHelper, fxParRT=\
158.1528, sfIX,  tnIX,  sfCvHDL, True,     False,   True,      [],     []

for kk, tt, bb in xcyBSS:
  if kk == 'fxsw':
    cHelper.append(ql.FxSwapRateHelper( sQH(bb/100),sQH(jpySP),pD(tt),
                              Tp2,calUJ,mFLLW,EoMf,baseCLLT,sfCvHDL) )
    fxParRT.append(bb/100)
  if kk == 'bss':
    cHelper.append( ql.MtMCrossCurrencyBasisSwapRateHelper(
                        sQH(bb*bps),pD(tt),Tp2,calUJ,mFLLW,EoMf,
                        baseIX,othrIX,clltCv,baseCLLT,baseBSS,baseRST,frqQ))
    fxParRT.append(bb*bps)
jBsCvOBJ = ql.PiecewiseLogLinearDiscount(Tp0,calUJ,cHelper,dcA365)
jBsCvHDL = ql.YieldTermStructureHandle(jBsCvOBJ)
mu.showCvDT(jBsCvOBJ); dfDSP(mu.dfNodes(jBsCvOBJ,fxParRT,cmpd=CNT),6)

tradeDT:2026-01-20, settleDT:2026-01-20 

,date,matYR,parRT,zeroRT,DF
0,2026-01-20,0.0000,nan%,0.601083%,1.00000000
1,2026-01-29,0.0247,-9.430000%,0.601083%,0.99985180
2,2026-02-24,0.0959,-44.850000%,0.578398%,0.99944553
3,2026-03-23,0.1699,-81.510000%,0.576547%,0.99902114
4,2026-04-22,0.2521,-121.850000%,0.569336%,0.99856599
5,2026-05-22,0.3342,-160.190000%,0.591828%,0.99802379
9,2027-01-22,1.0055,-433.700000%,0.724243%,0.99274434
10,2027-07-22,1.5014,-0.256250%,0.821466%,0.98774250
11,2028-01-24,2.0110,-0.281250%,0.915075%,0.98176649
12,2029-01-22,3.0082,-0.313750%,1.057407%,0.96869148


In [4]:
# hc: usd cllt jpy-1YDF
fwd1y = jpySP-433.70/100 ; sf1yDF = 0.96535756
print(f'fwd1y:{fwd1y:.4f}, (hc)cllt-1yDF:{jpySP/fwd1y*sf1yDF:.6f}')

fwd1y:153.8158, (hc)cllt-1yDF:0.992577
